# RAG Pipeline 拆解测试

本 Notebook 将 RAG（Retrieval-Augmented Generation）技术拆解为以下几个独立模块，以确保每一步可查且执行顺利：
1. 准备和环境加载
2. 读文件 (Document Loader)
3. 文本切分 (Chunking)
4. Embedding 与向量库存储 (Vector Database)
5. Query Embedding 与初步召回 (Retrieval)
6. 结果重排序 (Reranking)

In [ ]:
# ---------------- 0. 准备工作环境与加载环境变量 ----------------
# 如果没有下面库，取消注释后安装：
# !pip install -q chromadb langchain-community langchain-openai langchain-chroma sentence-transformers python-dotenv 

import os
from dotenv import load_dotenv

# 加载环境变量以获取需要的 API Key，例如 OPENAI_API_KEY
load_dotenv(dotenv_path="../.env")

print("OpenAI API Key 当前状态:", "已设置" if os.environ.get("OPENAI_API_KEY") else "未设置 (提醒: 这步需要key，否则后续embedding会报错)")
print("提示：如果报错，请确保项目根目录的 .env 文件中有对应的 OPENAI_API_KEY。")

### 1. 读文件 (Document Loading)
这里为了方便演示，我们在当前目录写入一个测试文档，然后用 LangChain 的 `TextLoader` 把它读进来。实际中可以是 `PyPDFLoader`，`UnstructuredWordDocumentLoader` 等。

In [ ]:
# ---------------- 1. 读取文件 ----------------
from langchain_community.document_loaders import TextLoader

# 构筑一份测试文件给 RAG 使用
sample_text = """
人工智能（Artificial Intelligence, AI）是计算机科学的一个分支，旨在了解智能的实质，并生产出一种新的能以人类智能相似的方式做出反应的智能机器。
大型语言模型（Large Language Models, LLM）是深度学习的重要进展，通过对海量的数据进行训练，具有了通识理解与强大的文本生成能力。
检索增强生成（Retrieval-Augmented Generation, RAG）是一种结合检索模型与生成模型的技术，通过从外部知识库检索出与查询相关的内容，显著提升生成的准确性和实时性。
向量数据库（Vector Database）如 Chroma 或 FAISS，专门用于大规模存储、管理和高效检索多维向量数据。
"""
file_path = "sample_knowledge.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(sample_text.strip())

# 模块 1: 用 Loader 独立读取
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print(f"成功读取文档数量: {len(documents)}")
print("---- 第一篇文档内容预览 ----")
print(documents[0].page_content[:150] + "...")

### 2. Chunking (文本切分)
大文件必须经过切分（Chunking），才能提取粒度合适的召回片段，并适应 Embedding 模型的最大长度或大模型的最终输入上下文。

In [ ]:
# ---------------- 2. Document Chunking ----------------
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 注：我们刻意将块大小设置得非常小（70个字符）以在这段较短的样例文本上充分展示分块效果
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=70,      # 每个分块最大字符数
    chunk_overlap=15    # 上下文重叠保留字符数，保证边界语义不断裂
)

chunked_docs = text_splitter.split_documents(documents)

print(f"原本的 {len(documents)} 篇文档被切分成了 {len(chunked_docs)} 个 chunk。\n")
for i, chunk in enumerate(chunked_docs):
    print(f"[Chunk {i+1}] (长度: {len(chunk.page_content)})")
    print(chunk.page_content)
    print("-" * 40)

### 3 & 4. Embedding 与 存入向量数据库
使用前述切好的 chunks，调用嵌入提取其特征向量，并存入轻量级本地向量库 Chroma 数据库。

In [ ]:
# ---------------- 3 & 4. Embedding与存入向量数据库 ----------------
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 模块 3: 构造 Embedding 对象
# 注意：如果你要用其他模型的 Embeddings（如 Zhipu, BGE等），可以在这一步里换成对应的 Embedding 类
embeddings_model = OpenAIEmbeddings()

# 模块 4: 初始化向量数据库并存入 Chunk 文档
# 直接将 chunks 和 emb_model 放入 Chroma，会在内存中建立索引（也可配置 persist_directory 持久化）
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings_model,
    collection_name="test_rag_collection"
)

print(f"成功将 {len(chunked_docs)} 个 chunks 转换为向量并存入 Chroma DB 中。")
print("此时由于存放在临时内存，如果内核重启库会自动清空。")

### 5 & 6. Query Embedding 与 首次召回 (Retrieval)
当用户提问时，也对其 Query 提取 embedding 并与数据库中的数据进行最相似度匹配和搜索。

In [ ]:
# ---------------- 5 & 6. Query Embedding 与 召回 ----------------
user_query = "什么是RAG技术？"

# 模块 5 (展示): 获取一条查询(query)的 Embedding 究竟长啥样
query_embedding = embeddings_model.embed_query(user_query)
print(f"提问 '{user_query}' 会被转换成长度为 {len(query_embedding)} 维的浮点数组，以此进行矩阵比对。\n")

# 模块 6: 召回最相似片段
# similarity_search 内部会先做刚才展示的 embed_query，随后在向量库里匹配 k 条最近的结果
top_k = 3
retrieved_docs = vector_store.similarity_search(user_query, k=top_k)

print(f"--- 初步向量召回 (Top {top_k}) 结果 ---")
for i, doc in enumerate(retrieved_docs):
    print(f"\n召回 Chunk {i+1}:\n{doc.page_content}")

### 7. 重排序 (Reranking)
基于向量相似度召回的内容容易产生“形似但神不似”的问题（无法完全反映逻辑和深层意图的相关性）。

这里我们通过 `CrossEncoder` 排序大模型或者其他重排服务计算 `(query, document)` pair 的混合特征得分，再进行精准重排。

In [ ]:
# ---------------- 7. Reranking 重排序 ----------------
try:
    from sentence_transformers import CrossEncoder
    import numpy as np
    
    # 加载免费轻量的跨级联模型进行打分。（注意：第一次运行需要部分网络下载缓存）
    print("正在加载 Reranker 模型（若首次执行请耐心等待下载）...")
    
    # 使用较小且效果不错的轻量级多语言 cross-encoder进行验证
    reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
    
    # 将 query 和 每一步召回打回的 document 内容进行打包配对
    pairs = [[user_query, doc.page_content] for doc in retrieved_docs]
    
    # 根据 pair 进行打重排序分, 这一步是模型真正在判定 两端文本在语义中互相回答的直接概率
    scores = reranker.predict(pairs)
    print("召回文档的新重排序对应分数:", scores)
    
    # 分数由高依次倒序排
    sorted_indices = np.argsort(scores)[::-1] 
    
    print("\n================ Reranker 重排序后的最终结果 ================")
    for new_rank, idx in enumerate(sorted_indices):
        original_rank = idx + 1
        score = scores[idx]
        print(f"【Rank {new_rank+1}】 (原召回名次: {original_rank}, Score: {score:.4f}):")
        print(retrieved_docs[idx].page_content)
        print("-" * 50)

except ImportError:
    print("[依赖提示]")
    print("环境未探测到 `sentence_transformers` 依赖，如果需要真实体验 Reranking 模块请执行：")
    print("!pip install sentence-transformers")
    print("\n--- 启动模拟重排演示 ---")
    print("[模拟] 对原召回文档进行逆序演示，旨在说明即使是Top召回也可能因为精细相关性的不同而改变位序:")
    for i, doc in enumerate(reversed(retrieved_docs)):
        print(f"【重排列 Reranked {i+1}】:\n{doc.page_content}\n")